# **7일차 실습: 도메인 특화 도구 만들기 및 테스트**

## 학습 목표
1. LangChain Tool 개념 이해
2. AI를 활용한 도구 자동 생성
3. 생성한 도구 테스트
4. 도메인 특화 에이전트에 적용

## 실습 단계
1. **도구 설계**: 팀 프로젝트에 필요한 도구 정의
2. **도구 생성**: AI를 활용하여 도구 코드 자동 생성
3. **도구 테스트**: 생성된 도구가 올바르게 동작하는지 확인
4. **에이전트 적용**: domain-agent에 통합 (다음 단계)

---

## 1. AI 도구 생성 프롬프트 확인

팀 프로젝트에 필요한 도구를 AI로 자동 생성하기 위한 프롬프트입니다.

**프롬프트 파일 위치**: `../../make_tool_prompt.txt`

In [1]:
# 프롬프트 파일 읽기
with open("../make_tool_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()

print("=" * 80)
print("AI 도구 생성 프롬프트")
print("=" * 80)
print(prompt_template)
print("\n✓ 프롬프트 로드 완료")
print("\n💡 이 프롬프트를 ChatGPT, Claude 등에 복사하여 사용하세요!")

AI 도구 생성 프롬프트
당신은 LangChain/LangGraph 기반 AI Agent를 위한 Python Tool 개발 전문가입니다.

아래 **[사용자 요구사항]** 에 작성된 내용을 바탕으로 LangChain Tool을 생성하세요.

---

# 사용자 요구사항

## 여기에 원하는 기능을 작성하세요.

<이곳에 원하는 Tool 기능을 작성>

---

# 구현 규칙

다음 규칙을 반드시 지키세요.

## 출력 형식

* Python 코드만 출력합니다.
* 코드 외의 설명은 출력하지 않습니다.
* `from langchain_core.tools import tool`을 사용합니다.
* `@tool(parse_docstring=True)` 데코레이터를 사용합니다.
* Python 3.11 이상 기준으로 작성합니다.

## 함수 작성 규칙

* 함수명은 기능에 맞게 작성합니다.
* 모든 매개변수에는 타입 힌트를 작성합니다.
* 반환 타입도 작성합니다.
* 필요한 import는 함수 내부에서 수행합니다.
* 가능한 표준 라이브러리를 우선 사용합니다.
* 외부 라이브러리가 필요한 경우 import도 함께 작성합니다.

## Docstring

반드시 Google Style Docstring을 작성합니다.

예시 형식

```python
"""도구 설명

Args:
    parameter1: 설명
    parameter2: 설명

Returns:
    반환값 설명
"""
```

## 예외 처리

반드시 예외 처리를 구현합니다.

형식

```python
try:
    ...
    return "성공 메시지"
except Exception as e:
    return f"실패: {str(e)}"
```

## 테스트 코드

마지막에 아래 코드를 추가합니다.

```python
print(f"도구 이름: {함수명.name}")
print(f"도구 설명: {함수명.description}")
```

## 코드 품질

* 읽기 쉬운 코드로 작성합니다.
* 

## 2. 도구 설계 가이드

### 좋은 도구의 조건

1. **단일 책임**: 하나의 명확한 기능만 수행
2. **명확한 입력/출력**: 매개변수와 반환값이 명확
3. **에러 처리**: 예외 상황을 적절히 처리
4. **좋은 설명**: Docstring으로 도구의 기능을 명확히 설명

### 도메인별 도구 예시

**쇼핑 도메인:**
- 상품 검색
- 가격 비교
- 재고 확인
- 리뷰 조회

**법령 도메인:**
- 법령 검색
- 조문 조회
- 판례 검색
- 법령 해석

**의료 도메인:**
- 증상 검색
- 병원 찾기
- 약 정보 조회
- 건강 정보 제공

**여행 도메인:**
- 항공권 검색
- 호텔 검색
- 관광지 정보
- 날씨 확인

---

## 3. 도구 생성 프로세스

### Step 1: 팀 프로젝트 도메인 및 필요한 도구 정의

**TODO: 팀에서 선택한 도메인과 필요한 도구를 작성하세요**

```
팀 도메인: [여기에 작성]

팀 도메인: 여행

필요한 도구 목록:
1. 도구명: 항공권 검색
   - 입력: 출발지, 도착지, 출발일
   - 출력: 항공사, 가격, 출발/도착 시간, 소요시간을 담은 항공편 목록
   - 역할: 사용자의 일정과 예산에 맞는 항공편 후보를 찾아 비교할 수 있게 해준다

2. 도구명: 호텔 검색
   - 입력: 도시, 체크인 날짜, 체크아웃 날짜, 인원수
   - 출력: 호텔명, 1박 가격, 평점, 위치를 담은 숙소 목록
   - 역할: 여행지에서 묵을 숙소 옵션을 예산/평점 기준으로 비교할 수 있게 해준다

3. 도구명: 관광지 정보 조회
   - 입력: 도시, 관심 카테고리(예: 맛집/명소/액티비티, 선택)
   - 출력: 추천 장소명, 간단 설명, 운영시간을 담은 리스트
   - 역할: 현지에서 무엇을 하고 어디를 가야 할지 여행 일정을 짜는 데 도움을 준다
```

### Step 2: AI로 도구 생성하기

**사용 방법:**

1. 위의 프롬프트 템플릿을 복사
2. `<이곳에 원하는 Tool 기능을 작성>` 부분에 팀의 도구 요구사항 작성
3. ChatGPT, Claude 등에 입력하여 코드 생성
4. 생성된 코드를 아래 셀에 붙여넣기

**예시 입력:**
```
쇼핑 도메인의 상품 검색 도구를 만들어주세요.

기능:
- 상품명으로 검색
- 가격 범위 필터링
- 카테고리 필터링
- 검색 결과를 JSON 형태로 반환
```

---

## 4. 생성된 도구 코드 테스트

**TODO: AI가 생성한 도구 코드를 아래에 붙여넣으세요**

**중요:** 
- 코드를 실행하기 전에 반드시 검토하세요
- 필요한 외부 라이브러리가 있다면 먼저 설치하세요
- 실제 API 키가 필요한 경우 .env 파일에 추가하세요

In [4]:
# TODO: AI가 생성한 도구 코드를 여기에 붙여넣으세요
from langchain_core.tools import tool


@tool(parse_docstring=True)
def search_flights(departure: str, destination: str, departure_date: str, max_price: int = 1000000) -> str:
    """출발지, 도착지, 날짜를 기준으로 항공권을 검색합니다.

    Args:
        departure: 출발 도시명 (예: '서울')
        destination: 도착 도시명 (예: '도쿄')
        departure_date: 출발 날짜 (YYYY-MM-DD 형식)
        max_price: 최대 가격(원). 기본값은 1,000,000원

    Returns:
        조건에 맞는 항공편 목록을 담은 JSON 문자열
    """
    import json

    try:
        flight_db = [
            {"airline": "대한항공", "departure": "서울", "destination": "도쿄", "date": "2026-09-01", "price": 320000, "duration": "2시간 30분"},
            {"airline": "아시아나항공", "departure": "서울", "destination": "도쿄", "date": "2026-09-01", "price": 280000, "duration": "2시간 40분"},
            {"airline": "제주항공", "departure": "서울", "destination": "오사카", "date": "2026-09-05", "price": 210000, "duration": "2시간 20분"},
            {"airline": "대한항공", "departure": "서울", "destination": "뉴욕", "date": "2026-10-10", "price": 1450000, "duration": "14시간 30분"},
        ]

        results = [
            f for f in flight_db
            if f["departure"] == departure
            and f["destination"] == destination
            and f["date"] == departure_date
            and f["price"] <= max_price
        ]

        if not results:
            return json.dumps({"message": "조건에 맞는 항공편이 없습니다.", "results": []}, ensure_ascii=False)

        return json.dumps({"message": f"{len(results)}건의 항공편을 찾았습니다.", "results": results}, ensure_ascii=False)
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """도시와 체크인/체크아웃 날짜를 기준으로 호텔을 검색합니다.

    Args:
        city: 숙박할 도시명 (예: '도쿄')
        check_in: 체크인 날짜 (YYYY-MM-DD 형식)
        check_out: 체크아웃 날짜 (YYYY-MM-DD 형식)
        guests: 투숙 인원수. 기본값은 1명

    Returns:
        조건에 맞는 호텔 목록을 담은 JSON 문자열
    """
    import json

    try:
        hotel_db = [
            {"name": "도쿄 스테이션 호텔", "city": "도쿄", "price_per_night": 180000, "rating": 4.5, "location": "도쿄역 인근"},
            {"name": "신주쿠 파크 호텔", "city": "도쿄", "price_per_night": 120000, "rating": 4.1, "location": "신주쿠"},
            {"name": "오사카 난바 호텔", "city": "오사카", "price_per_night": 95000, "rating": 4.0, "location": "난바"},
            {"name": "맨해튼 센트럴 호텔", "city": "뉴욕", "price_per_night": 350000, "rating": 4.3, "location": "맨해튼"},
        ]

        if check_in >= check_out:
            return f"실패: 체크아웃 날짜({check_out})는 체크인 날짜({check_in})보다 이후여야 합니다."

        results = [h for h in hotel_db if h["city"] == city]

        if not results:
            return json.dumps({"message": f"'{city}'에 등록된 호텔이 없습니다.", "results": []}, ensure_ascii=False)

        return json.dumps(
            {
                "message": f"{len(results)}건의 호텔을 찾았습니다. (체크인: {check_in}, 체크아웃: {check_out}, 인원: {guests}명)",
                "results": results,
            },
            ensure_ascii=False,
        )
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def get_attractions(city: str, category: str = "") -> str:
    """도시와 관심 카테고리를 기준으로 관광지 정보를 조회합니다.

    Args:
        city: 조회할 도시명 (예: '도쿄')
        category: 관심 카테고리 (예: '맛집', '명소', '액티비티'). 지정하지 않으면 전체 반환

    Returns:
        추천 장소 목록을 담은 JSON 문자열
    """
    import json

    try:
        attraction_db = [
            {"city": "도쿄", "category": "명소", "name": "센소지", "description": "아사쿠사의 대표 사찰", "hours": "06:00-17:00"},
            {"city": "도쿄", "category": "맛집", "name": "츠키지 장외시장", "description": "신선한 해산물 맛집 거리", "hours": "05:00-14:00"},
            {"city": "도쿄", "category": "액티비티", "name": "팀랩 플래닛", "description": "디지털 아트 체험 전시", "hours": "10:00-21:00"},
            {"city": "오사카", "category": "명소", "name": "오사카성", "description": "오사카를 대표하는 성", "hours": "09:00-17:00"},
        ]

        results = [a for a in attraction_db if a["city"] == city and (not category or a["category"] == category)]

        if not results:
            return json.dumps({"message": f"'{city}' ({category or '전체'}) 조건에 맞는 정보가 없습니다.", "results": []}, ensure_ascii=False)

        return json.dumps({"message": f"{len(results)}건의 장소를 찾았습니다.", "results": results}, ensure_ascii=False)
    except Exception as e:
        return f"실패: {str(e)}"


print(f"도구 이름: {search_flights.name}")
print(f"도구 설명: {search_flights.description}")
print(f"도구 이름: {search_hotels.name}")
print(f"도구 설명: {search_hotels.description}")
print(f"도구 이름: {get_attractions.name}")
print(f"도구 설명: {get_attractions.description}")

도구 이름: search_flights
도구 설명: 출발지, 도착지, 날짜를 기준으로 항공권을 검색합니다.
도구 이름: search_hotels
도구 설명: 도시와 체크인/체크아웃 날짜를 기준으로 호텔을 검색합니다.
도구 이름: get_attractions
도구 설명: 도시와 관심 카테고리를 기준으로 관광지 정보를 조회합니다.


## 5. 도구 정보 확인

생성된 도구의 메타데이터를 확인합니다.

In [5]:
# TODO: 생성한 도구의 함수명으로 변경하세요
tool_function_name = search_flights

print("=" * 80)
print("도구 정보")
print("=" * 80)
print(f"도구 이름: {tool_function_name.name}")
print(f"도구 설명: {tool_function_name.description}")
print(f"\n입력 스키마:")
print(tool_function_name.args_schema.schema())
pass  # 위 주석을 해제하고 사용하세요

도구 정보
도구 이름: search_flights
도구 설명: 출발지, 도착지, 날짜를 기준으로 항공권을 검색합니다.

입력 스키마:
{'description': '출발지, 도착지, 날짜를 기준으로 항공권을 검색합니다.', 'properties': {'departure': {'description': "출발 도시명 (예: '서울')", 'title': 'Departure', 'type': 'string'}, 'destination': {'description': "도착 도시명 (예: '도쿄')", 'title': 'Destination', 'type': 'string'}, 'departure_date': {'description': '출발 날짜 (YYYY-MM-DD 형식)', 'title': 'Departure Date', 'type': 'string'}, 'max_price': {'default': 1000000, 'description': '최대 가격(원). 기본값은 1,000,000원', 'title': 'Max Price', 'type': 'integer'}}, 'required': ['departure', 'destination', 'departure_date'], 'title': 'search_flights', 'type': 'object'}


C:\Users\kji49\AppData\Local\Temp\ipykernel_20428\2620146249.py:10: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(tool_function_name.args_schema.schema())


## 6. 도구 단독 실행 테스트

**TODO: 다양한 입력값으로 도구를 테스트하세요**

테스트 케이스를 최소 3개 이상 작성하세요:
1. 정상 케이스
2. 엣지 케이스 (경계값)
3. 에러 케이스 (잘못된 입력)

In [6]:
# TODO: 도구를 직접 호출하여 테스트하세요

print("테스트 1: 정상 케이스 (서울 -> 도쿄, 2026-09-01)")
result1 = search_flights.invoke({
    "departure": "서울",
    "destination": "도쿄",
    "departure_date": "2026-09-01",
})
print(result1)
print()

print("테스트 2: 엣지 케이스 (가격 상한을 정확히 최저가에 맞춤 -> 1건만 남아야 함)")
result2 = search_flights.invoke({
    "departure": "서울",
    "destination": "도쿄",
    "departure_date": "2026-09-01",
    "max_price": 280000,
})
print(result2)
print()

print("테스트 3: 에러 케이스 (존재하지 않는 도시)")
result3 = search_flights.invoke({
    "departure": "부산",
    "destination": "런던",
    "departure_date": "2026-09-01",
})
print(result3)
print()

print("테스트 4: search_hotels 정상 케이스 (도쿄, 1박)")
result4 = search_hotels.invoke({
    "city": "도쿄",
    "check_in": "2026-09-01",
    "check_out": "2026-09-02",
    "guests": 2,
})
print(result4)
print()

print("테스트 5: search_hotels 에러 케이스 (체크아웃이 체크인보다 빠름)")
result5 = search_hotels.invoke({
    "city": "도쿄",
    "check_in": "2026-09-05",
    "check_out": "2026-09-01",
})
print(result5)
print()

print("테스트 6: get_attractions 정상 케이스 (도쿄, 명소)")
result6 = get_attractions.invoke({"city": "도쿄", "category": "명소"})
print(result6)
print()

print("테스트 7: get_attractions 엣지 케이스 (카테고리 미지정 -> 도쿄 전체)")
result7 = get_attractions.invoke({"city": "도쿄"})
print(result7)

pass  # 위 주석을 해제하고 사용하세요

테스트 1: 정상 케이스 (서울 -> 도쿄, 2026-09-01)
{"message": "2건의 항공편을 찾았습니다.", "results": [{"airline": "대한항공", "departure": "서울", "destination": "도쿄", "date": "2026-09-01", "price": 320000, "duration": "2시간 30분"}, {"airline": "아시아나항공", "departure": "서울", "destination": "도쿄", "date": "2026-09-01", "price": 280000, "duration": "2시간 40분"}]}

테스트 2: 엣지 케이스 (가격 상한을 정확히 최저가에 맞춤 -> 1건만 남아야 함)
{"message": "1건의 항공편을 찾았습니다.", "results": [{"airline": "아시아나항공", "departure": "서울", "destination": "도쿄", "date": "2026-09-01", "price": 280000, "duration": "2시간 40분"}]}

테스트 3: 에러 케이스 (존재하지 않는 도시)
{"message": "조건에 맞는 항공편이 없습니다.", "results": []}

테스트 4: search_hotels 정상 케이스 (도쿄, 1박)
{"message": "2건의 호텔을 찾았습니다. (체크인: 2026-09-01, 체크아웃: 2026-09-02, 인원: 2명)", "results": [{"name": "도쿄 스테이션 호텔", "city": "도쿄", "price_per_night": 180000, "rating": 4.5, "location": "도쿄역 인근"}, {"name": "신주쿠 파크 호텔", "city": "도쿄", "price_per_night": 120000, "rating": 4.1, "location": "신주쿠"}]}

테스트 5: search_hotels 에러 케이스 (체크아웃이 체크인보다 빠름)
실패: 체크

## 7. 여러 도구 통합 테스트

팀에서 만든 여러 도구를 함께 테스트합니다.

**TODO: 생성한 모든 도구를 리스트로 정리하세요**

In [7]:
# TODO: 팀에서 생성한 모든 도구를 리스트로 추가하세요

CUSTOM_TOOLS = [
    search_flights,
    search_hotels,
    get_attractions,
]

print(f"총 {len(CUSTOM_TOOLS)}개의 도구가 준비되었습니다.\n")

for i, t in enumerate(CUSTOM_TOOLS, 1):
    print(f"{i}. {t.name}")
    print(f"   설명: {t.description}")
    print()

pass  # 위 주석을 해제하고 사용하세요

총 3개의 도구가 준비되었습니다.

1. search_flights
   설명: 출발지, 도착지, 날짜를 기준으로 항공권을 검색합니다.

2. search_hotels
   설명: 도시와 체크인/체크아웃 날짜를 기준으로 호텔을 검색합니다.

3. get_attractions
   설명: 도시와 관심 카테고리를 기준으로 관광지 정보를 조회합니다.



## 프로젝트 체크리스트

**완료한 항목을 확인하세요:**

- [ ] 팀 도메인 선정 및 필요한 도구 정의 완료
- [ ] AI 프롬프트를 사용하여 도구 코드 생성 완료
- [ ] 최소 3개 이상의 도구 생성 완료
- [ ] 각 도구별 단독 실행 테스트 완료
- [ ] 정상/엣지/에러 케이스 테스트 완료
- [ ] 도구 메타데이터 확인 완료

---

## 다음 단계

생성한 도구를 domain-agent에 통합하세요:

1. `../src/domain-agent/tools.py` 파일 열기
2. TODO 주석을 참고하여 생성한 도구 코드 추가
3. `../src/domain-agent/agent.py` 파일 열기
4. TODO 주석을 참고하여 시스템 프롬프트와 도구 리스트 수정
5. LangGraph Studio로 테스트

---

## 참고 자료

- [LangChain Tools 문서](https://python.langchain.com/docs/modules/agents/tools/)
- [LangChain Custom Tools](https://python.langchain.com/docs/modules/agents/tools/custom_tools/)
- [@tool 데코레이터](https://python.langchain.com/docs/modules/agents/tools/custom_tools/#tool-decorator)